<a href="https://colab.research.google.com/github/sungtho759-ctrl/cat-dog-project/blob/main/catsdogspj.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Lệnh này sẽ giải nén file zip từ Drive thẳng vào ổ cứng siêu tốc của Colab
!unzip -q '/content/drive/MyDrive/MyProject/cat-dog-project.zip' -d '/content/'

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
import os

# Đường dẫn đến thư mục dataset trên Drive của bạn
base_dir = '/content/cat-dog-project'
train_dir = os.path.join(base_dir, 'train')
validation_dir = os.path.join(base_dir, 'val')

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
import os

# 1. Đường dẫn thư mục (Sửa lại cho đúng với Drive của bạn)
base_dir = '/content/cat-dog-project'
train_dir = os.path.join(base_dir, 'train')
validation_dir = os.path.join(base_dir, 'val')

# 2. Xử lý dữ liệu chuẩn cho MobileNetV2 (KHÔNG dùng rescale=1./255)
train_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_datagen.flow_from_directory(
    train_dir, target_size=(224, 224), batch_size=32, class_mode='binary')

validation_generator = val_datagen.flow_from_directory(
    validation_dir, target_size=(224, 224), batch_size=32, class_mode='binary')

# 3. Xây dựng mô hình với MobileNetV2
base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False # Đóng băng các lớp cơ sở

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

# 4. Biên dịch và Huấn luyện
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
              loss='binary_crossentropy',
              metrics=['accuracy'])

print("Bắt đầu huấn luyện...")
history = model.fit(
    train_generator,
    epochs=5, # Train 5 vòng là đủ thông minh rồi
    validation_data=validation_generator
)

# 5. Lưu mô hình chuẩn
model.save('dog-cat-mobilenet.keras')
print("Đã lưu mô hình thành công!")

Found 22441 images belonging to 2 classes.
Found 2598 images belonging to 2 classes.
Bắt đầu huấn luyện...
Epoch 1/5
142/702 ━━━━━━━━━━━━━━━━━━━━ 59s 106ms/step - accuracy: 0.8349 - loss: 0.3603

/usr/local/lib/python3.13/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


702/702 ━━━━━━━━━━━━━━━━━━━━ 114s 142ms/step - accuracy: 0.9681 - loss: 0.0909 - val_accuracy: 0.9885 - val_loss: 0.0358
Epoch 2/5
702/702 ━━━━━━━━━━━━━━━━━━━━ 88s 125ms/step - accuracy: 0.9842 - loss: 0.0434 - val_accuracy: 0.9911 - val_loss: 0.0306
Epoch 3/5
702/702 ━━━━━━━━━━━━━━━━━━━━ 92s 130ms/step - accuracy: 0.9869 - loss: 0.0367 - val_accuracy: 0.9873 - val_loss: 0.0307
Epoch 4/5
702/702 ━━━━━━━━━━━━━━━━━━━━ 88s 126ms/step - accuracy: 0.9884 - loss: 0.0327 - val_accuracy: 0.9919 - val_loss: 0.0279
Epoch 5/5
702/702 ━━━━━━━━━━━━━━━━━━━━ 84s 120ms/step - accuracy: 0.9904 - loss: 0.0282 - val_accuracy: 0.9915 - val_loss: 0.0274
Đã lưu mô hình thành công!


In [ ]:
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(224, 224, 3)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),  # 👈 thêm tầng
    MaxPooling2D(2,2),

    tf.keras.layers.GlobalAveragePooling2D(),  # 👈 thay Flatten

    Dense(256, activation='relu'),
    Dropout(0.5),

    Dense(1, activation='sigmoid')
])


/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [ ]:
model.compile(
    loss='binary_crossentropy',
    optimizer=tf.keras.optimizers.RMSprop(learning_rate=1e-4),
    metrics=['accuracy']
)


In [ ]:
model.save('dog-cat-project.h5')

In [ ]:
history = model.fit(
    train_generator,
    epochs=40,
    validation_data=validation_generator,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/40
702/702 ━━━━━━━━━━━━━━━━━━━━ 88s 126ms/step - accuracy: 0.9933 - loss: 0.0208 - val_accuracy: 0.9904 - val_loss: 0.0278
Epoch 2/40
702/702 ━━━━━━━━━━━━━━━━━━━━ 88s 125ms/step - accuracy: 0.9938 - loss: 0.0184 - val_accuracy: 0.9888 - val_loss: 0.0299
Epoch 3/40
702/702 ━━━━━━━━━━━━━━━━━━━━ 84s 119ms/step - accuracy: 0.9945 - loss: 0.0166 - val_accuracy: 0.9919 - val_loss: 0.0273
Epoch 4/40
702/702 ━━━━━━━━━━━━━━━━━━━━ 87s 124ms/step - accuracy: 0.9944 - loss: 0.0158 - val_accuracy: 0.9923 - val_loss: 0.0286
Epoch 5/40
702/702 ━━━━━━━━━━━━━━━━━━━━ 89s 127ms/step - accuracy: 0.9955 - loss: 0.0148 - val_accuracy: 0.9923 - val_loss: 0.0285


In [ ]:
import gradio as gr
import tensorflow as tf
import numpy as np
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# 1. Load mô hình mới mà bạn vừa train xong
model = tf.keras.models.load_model('dog-cat-mobilenet.keras')

def predict_image(img):
    # 2. Resize ảnh về 224x224 (bắt buộc phải khớp với lúc train)
    img = img.resize((224, 224))

    # 3. Chuyển thành mảng và mở rộng chiều
    img_array = np.array(img)
    img_array = np.expand_dims(img_array, axis=0)

    # 4. Tiền xử lý ảnh ĐÚNG CHUẨN MobileNetV2 (Không dùng chia 255 nữa)
    img_array = preprocess_input(img_array)

    # 5. Dự đoán
    prediction = model.predict(img_array)
    score = float(prediction[0][0])

    # Mặc định trong Keras (nếu thư mục tên là 'cats' và 'dogs'):
    # C đứng trước D trong bảng chữ cái, nên 'cats' = 0, 'dogs' = 1
    # Nếu score > 0.5 nghĩa là thiên về 1 (Chó)
    if score > 0.5:
        return {"Chó": score, "Mèo": 1 - score}
    else:
        return {"Mèo": 1 - score, "Chó": score}

# 6. Tạo giao diện
demo = gr.Interface(
    fn=predict_image,
    inputs=gr.Image(type="pil"),
    outputs=gr.Label(num_top_classes=2),
    title="AI Nhận Diện Chó/Mèo (Bản Chuẩn MobileNetV2)"
)

# Chạy tạo link web
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c8c8707988a679efe5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
